

# Detección de fraude (tarjetas de crédito)

En este estudio de caso se usan distintos modelos de **clasificación** para predecir si una transacción es un pago legítimo o un **fraude**. El objetivo es entrenar, comparar y afinar algoritmos; la razón de usar varios enfoques es que el desbalance extremo de clases hace que la *exactitud* sola no baste y haya que elegir métricas y estrategias (p. ej. *recall* o remuestreo) alineadas con el coste de dejar pasar un fraude frente a una falsa alarma.


## Contenido


* [1. Definición del problema](#0)
* [2. Inicio: bibliotecas y conjunto de datos](#1)
    * [2.1. Carga de bibliotecas](#1.1)    
    * [2.2. Carga de datos](#1.2)
* [3. Análisis exploratorio (EDA)](#2)
    * [3.1 Estadística descriptiva](#2.1)    
    * [3.2. Visualización](#2.2)
* [4. Preparación de los datos](#3)
    * [4.1 Limpieza de datos](#3.1)    
    * [4.2. Selección de variables (features)](#3.2) 
* [5. Evaluación de algoritmos y modelos](#4)        
    * [5.1. Partición entrenamiento / validación](#4.1)
    * [5.2. Comparación de modelos](#4.2) 
* [6. Ajuste (tuning) del modelo](#5)
    * [6.1. Elegir la métrica de evaluación adecuada](#5.1)
    * [6.2. Balanceo del conjunto (submuestreo aleatorio)](#5.2) 



<a id='0'></a>
# 1. Definición del problema


Nos planteamos un problema de **clasificación binaria**.

- **Variable respuesta** `Class`: toma el valor **1** si la transacción es fraudulenta y **0** en caso contrario. El éxito del negocio suele medirse en términos de detectar el mayor número posible de fraudes, no solo de acierto global.

El conjunto de referencia reúne transacciones con tarjeta de crédito (septiembre de 2013, titulares en Europa), concentradas en **dos días** de registro, con alrededor de **492** fraudes entre **284 807** transacciones. La clase minoritaria (fraude) representa aproximadamente **0,172 %** del total, es decir, los datos están **muy desbalanceados**: un modelo que predijera “siempre no fraude” tendría exactitud casi del 100 % y aun así sería inútil.

Las **variables de entrada** `V1`–`V28` son el resultado de una transformación (en la práctica, componentes de tipo PCA aplicados por confidencialidad; por eso **no** tienen nombres interpretables directamente). Quedan además `Time` (tiempo transcurrido respecto a la primera transacción), `Amount` (importe) y `Class` como etiqueta. **No** importa aquí de dónde se descargó el CSV; basta con tener el fichero (completo o una muestra) en la misma carpeta que el notebook.


<a id='1'></a>
# 2. Inicio: carga de datos y paquetes de Python

En esta sección se instalan (mediante `import`) las herramientas necesarias: manipulación de tablas, gráficos, `scikit-learn` para modelos y validación, y opcionalmente `Keras` para una red profunda. La razón es **centralizar** el flujo: mismos tipos de datos, mismas funciones de partición y mismas métricas en todo el notebook.


<a id='1.1'></a>
## 2.1. Carga de las bibliotecas de Python

Se importan NumPy y Pandas para arrays y tablas; Matplotlib y Seaborn para EDA; `sklearn` para preprocesado, modelos, validación cruzada, búsqueda de hiperparámetros y métricas; y componentes de Keras para el clasificador neuronal opcional. Se incluye `pickle` para poder **guardar y cargar** un modelo entrenado más adelante si hace falta desplegarlo.


In [ ]:
# Carga de bibliotecas: números, tablas, gráficos y aprendizaje automático
import numpy as np
import pandas as pd
from matplotlib import pyplot
from pandas import read_csv, set_option
from pandas.plotting import scatter_matrix
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Keras: modelos de aprendizaje profundo (uso opcional más abajo)
from keras.models import Sequential
from keras.layers import Dense
from keras.wrappers.scikit_learn import KerasClassifier
from keras.optimizers import SGD

# Serialización del modelo (guardar / cargar)
from pickle import dump
from pickle import load


<a id='1.2'></a>
## 2.2. Carga de los datos

Aquí se **leen** filas y columnas desde un archivo CSV. El paso es obligatorio antes de EDA, partición y entrenamiento; la razón es que Pandas carga un `DataFrame` en memoria con tipos de datos unificados y nombres de columnas listos para `train_test_split` y los estimadores de `sklearn`.


In [ ]:
# Lectura del conjunto: ajusta el nombre de archivo a tu entorno (muestra o conjunto completo)
dataset = read_csv('creditcard_sample.csv')
# dataset = read_csv('creditcard.csv')  # descomentar para usar el fichero con todas las filas


In [ ]:
# Suprime avisos (warnings) poco informativos durante entrenamiento y validación
import warnings
warnings.filterwarnings('ignore')


<a id='2'></a>
# 3. Análisis exploratorio de los datos (EDA)

Antes de modelar, se **examina** el conjunto: forma, tipos, estadísticos, balance de clases y forma de las distribuciones. La razón es detectar fuga de datos, columnas inútiles, escala de variables y el **desbalanceo**, que condiciona la métrica y la estrategia (p. ej. submuestreo o *recall*).


<a id='2.1'></a>
## 3.1. Estadística descriptiva

Primero se mira **cuántas** filas y columnas hay (`shape`); sirve asegurar que el fichero se cargó bien y a dimensionar memoria y tiempo de entrenamiento.


In [ ]:
# Dimensiones: (número de filas, número de columnas)
dataset.shape


In [ ]:
# Primeras filas: estructura y orden de magnitud de las variables
set_option('display.width', 100)
dataset.head(5)


In [ ]:
# Tipo de cada columna: numérico entero/continuo, para saber qué preprocesado hace falta
set_option('display.max_rows', 500)
dataset.dtypes


Como se ve, casi todo es **flotante**; la variable `Class` es **entera** (0/1). Los nombres `V1`… no son explicables por su etiqueta, pero sí por su comportamiento en correlación y en modelos.


In [ ]:
# Resumen (media, desviación, mín., máx., cuartiles) para detectar escalas y valores atípicos
set_option('precision', 3)
dataset.describe()


Conteo de transacciones **fraudulentas** frente a **legítimas**. Este paso es clave: justifica por qué un modelo de “máxima exactitud” puede ignorar al fraude. La **razón** de mirar frecuencias es cuantificar el desbalanceo antes de particionar o remuestrear.


In [ ]:
nombres_clase = {0: 'No fraude', 1: 'Fraude'}
print(dataset.Class.value_counts().rename(index=nombres_clase))


El conjunto está **muy desbalanceado**: la gran mayoría de filas son “no fraude”. Por eso, en las secciones posteriores se cambia de métrica y se prueba el balanceo: no es un detalle estético, sino el núcleo del problema de fraude con datos reales.


<a id='2.2'></a>
## 3.2. Visualización

Histogramas por columna: ayudan a ver **asimetría** y colas pesadas. La razón de graficar es decidir si hace falta escalado, transformaciones o modelos que toleren escalas dispares; con nombres anonimizados, la intuición sustitutiva es la forma de la distribución, no el significado de “V7”.


In [ ]:
# Histogramas: distribución univariante de cada variable
dataset.hist(sharex=False, sharey=False, xlabelsize=1, ylabelsize=1, figsize=(12, 12))
pyplot.show()


Muchas distribuciones se ven **muy asimétricas**. Dado el anonimato de `V*`, el gráfico aporta señal de **comportamiento** (colas, concentración en cero) más que de “sentido de negocio” literal.


<a id='3'></a>
## 4. Preparación de los datos

Se comprueba calidad básica (nulos) y, si aplica, se eligen variables más informativas. **Motivación:** los algoritmos de `sklearn` no admiten celdas vacías de forma directa, y reducir ruido en *features* puede bajar varianza; aquí el conjunto ya viene en formato numérico, lo que acelera el flujo.


In [ ]:
# Comprueba si hay valores nulos (faltantes)
print('¿Hay valores nulos? =', dataset.isnull().values.any())


En este dataset **no** aparecen nulos y las entradas son **numéricas** de partida, así que no hace falta en este capítulo una codificación categórica (one-hot) ni un imputado masivo. La razón de saltar “limpieza” pesada es **ganar** claridad: el cuello de botella no es el formato, sino el desbalanceo y la métrica.


<a id='3.2'></a>
## 4.2. Selección de variables (prueba estadística univariada)

Se usa `SelectKBest` con puntuación tipo **chi-cuadrado** (aplicable a tablas de conteo; aquí se usa de forma ilustrativa sobre variables numéricas discretizadas de facto por el algoritmo) para listar candidatos “más asociados” a `Class`. **Propósito:** priorizar *features* si hubiera muchas redundantes; en este flujo se enfatiza el concepto, no un filtrado agresivo del conjunto al completo (las columnas anónimas suelen aprovecharse todas).


In [ ]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2

mejores = SelectKBest(k=10)
Y = dataset['Class']
X = dataset.loc[:, dataset.columns != 'Class']
ajuste = mejores.fit(X, Y)
dfscores = pd.DataFrame(ajuste.scores_)
dfcolumns = pd.DataFrame(X.columns)
puntuacion_variables = pd.concat([dfcolumns, dfscores], axis=1)
puntuacion_variables.columns = ['Variable', 'Puntuación']
print(puntuacion_variables.nlargest(10, 'Puntuación'))


Algunas variables reciben puntuación más alta, pero en este recorrido **no** se fuerza un modelo basado solo en un subconjunto: el conjunto completo de entradas numéricas se mantiene en los pasos posteriores para no perder señal repartida en muchas `V*`. La *razón* de ser cautos es la dependencia multivariante (lo que mide chi-cuadrado de forma aislada no captura interacciones).


<a id='4'></a>
# 5. Evaluación de algoritmos y modelos

A partir de aquí se **separa** un trozo de datos para *validar* sin tocar el entrenamiento, se fija una **métrica** inicial (exactitud) y se comparan algoritmos. La *razón* de la partición con `test_size=0.2` y `random_state` fijado es **estimación honesta** del error de generalización y **reproducibilidad**.


<a id='4.1'></a>
## 5.1. Partición entrenamiento / validación y métricas

Se definen `X` (sin `Class`) e `Y` (solo etiqueta). `train_test_split` deja aprox. un **20 %** para validar al final. Se usa al principio `scoring='accuracy'`: se verá enseguida que, con clases desbalanceadas, la exactitud **engaña** si el interés real es cazar fraude.


In [ ]:
# Reserva al final un 20% para validación (misma semilla para reproducir resultados)
Y = dataset['Class']
X = dataset.loc[:, dataset.columns != 'Class']
validation_size = 0.2
seed = 7
X_train, X_validation, Y_train, Y_validation = train_test_split(
    X, Y, test_size=validation_size, random_state=seed
)
scoring = 'accuracy'


<a id='4.2'></a>
## 5.2. Prueba de varios algoritmos y modelos (validación cruzada)

Se define un **10-fold** `KFold` y, para varios modelos, `cross_val_score` sobre el **conjunto de entrenamiento** con la métrica elegida. **Por qué:** promediar plegados suaviza la varianza respecto a un solo split. Parte de los algoritmos costosos quedan comentados cuando el conjunto es grande, para ahorrar tiempo; puedes descomentarlos si el hardware lo permite.


In [ ]:
# Opciones de validación cruzada: 10 pliegues, semilla fija
num_folds = 10
seed = 7


In [ ]:
# Comparación inicial de algoritmos (los más lentos o pesados quedan comentados con conjuntos enormes)
models = []
models.append(('RL', LogisticRegression()))
models.append(('LDA', LinearDiscriminantAnalysis()))
models.append(('K-vecinos', KNeighborsClassifier()))
models.append(('CART / árbol', DecisionTreeClassifier()))
# models.append(('NB', GaussianNB()))
# models.append(('SVM', SVC()))
# models.append(('Red neuronal MLP', MLPClassifier()))
# models.append(('AdaBoost', AdaBoostClassifier()))
# models.append(('GBM', GradientBoostingClassifier()))
# models.append(('Bosque aleatorio', RandomForestClassifier()))
# models.append(('Extra árboles', ExtraTreesClassifier()))


In [ ]:
resultados = []
nombres = []
for nombre, modelo in models:
    kfold = KFold(n_splits=num_folds, random_state=seed)
    puntuaciones_cv = cross_val_score(
        modelo, X_train, Y_train, cv=kfold, scoring=scoring
    )
    resultados.append(puntuaciones_cv)
    nombres.append(nombre)
    msg = '%s: media=%f (desv.=%f)' % (nombre, puntuaciones_cv.mean(), puntuaciones_cv.std())
    print(msg)


In [ ]:
# Diagrama de cajas: dispersión de la métrica entre pliegues para cada algoritmo
fig = pyplot.figure()
fig.suptitle('Comparación de algoritmos (validación cruzada)')
ax = fig.add_subplot(111)
pyplot.boxplot(resultados)
ax.set_xticklabels(nombres)
fig.set_size_inches(8, 4)
pyplot.show()


La **exactitud** es muy alta en términos globales porque el modelo puede acertar “no fraude” casi siempre. Eso no garantiza un buen desempeño **sobre la clase fraude**; el siguiente análisis toma un modelo razonable de la comparación (árbol, `CART`) y mira la **matriz de confusión** y el informe por clase. **Razón del cambio de foco:** en fraude, el coste de un **falso negativo** (fraude no detectado) suele ser mayor que el de un falso positivo, al menos para la primera capa de alertas.


In [ ]:
# Entrena un árbol de decisión con el set de entrenamiento
model = DecisionTreeClassifier()
model.fit(X_train, Y_train)


In [ ]:
# Evalúa en el conjunto de validación reservado (sin escalado: las variables ya están en formas similares)
X_val_escalada = X_validation
predicciones = model.predict(X_val_escalada)
print(accuracy_score(Y_validation, predicciones))
print(confusion_matrix(Y_validation, predicciones))
print(classification_report(Y_validation, predicciones))


In [ ]:
df_cm = pd.DataFrame(
    confusion_matrix(Y_validation, predicciones),
    columns=np.unique(Y_validation),
    index=np.unique(Y_validation)
)
df_cm.index.name = 'Real'
df_cm.columns.name = 'Predicción'
sns.heatmap(df_cm, cmap='Blues', annot=True, annot_kws={'size': 16})


Aunque los números globales parezcan aceptables, en la matriz de confusión aún quedan **falsos negativos** en fraude: muchos de los fraudes reales en validación no se marcan. Por tanto conviene acentuar el **recall** (sensibilidad) sobre el fraude, es decir, **minimizar** los no detectados, aunque suban las alarmas en legítimas. Esa es la *razón* de pasar a `scoring='recall'` en la siguiente fase.


<a id='5'></a>
## 6. Ajuste del modelo (tuning)

Aquí se **re-optimiza** el criterio: ya no basta con la exactitud. Se repite el barrido con la misma lista de algoritmos pero midiendo *recall* en validación cruzada.


<a id='5.1'></a>
## 6.1. Ajuste eligiendo la métrica: *recall* (sensibilidad al fraude)

Se fija `scoring = 'recall'`. **Motivación:** en detección de fraude, a menudo se prioriza **no dejar escapar** transacciones sospechosas; el *recall* de la clase positiva mide qué fracción de fraudes reales se detecta. Complementariamente luego se vigilan falsos positivos (coste operativo de revisión).


In [ ]:
scoring = 'recall'


In [ ]:
# Comparación inicial de algoritmos (los más lentos o pesados quedan comentados con conjuntos enormes)
models = []
models.append(('RL', LogisticRegression()))
models.append(('LDA', LinearDiscriminantAnalysis()))
models.append(('K-vecinos', KNeighborsClassifier()))
models.append(('CART / árbol', DecisionTreeClassifier()))
# models.append(('NB', GaussianNB()))
# models.append(('SVM', SVC()))
# models.append(('Red neuronal MLP', MLPClassifier()))
# models.append(('AdaBoost', AdaBoostClassifier()))
# models.append(('GBM', GradientBoostingClassifier()))
# models.append(('Bosque aleatorio', RandomForestClassifier()))
# models.append(('Extra árboles', ExtraTreesClassifier()))


In [ ]:
resultados = []
nombres = []
for nombre, modelo in models:
    kfold = KFold(n_splits=num_folds, random_state=seed)
    puntuaciones_cv = cross_val_score(
        modelo, X_train, Y_train, cv=kfold, scoring=scoring
    )
    resultados.append(puntuaciones_cv)
    nombres.append(nombre)
    msg = '%s: media=%f (desv.=%f)' % (nombre, puntuaciones_cv.mean(), puntuaciones_cv.std())
    print(msg)


Según el *recall* promedio en cruce de validación, el modelo **LDA** (análisis discriminante lineal) encabeza el grupo; se usa para ajuste final sobre entrenamiento y evaluación en el **mismo** conjunto de validación que antes, de modo comparable.


In [ ]:
# Entrenamiento: LDA (Linear Discriminant Analysis)
model = LinearDiscriminantAnalysis()
model.fit(X_train, Y_train)


In [ ]:
# Validación: mismas entradas que con el árbol (coherencia con el preprocesado usado: ninguno adicional)
X_val_escalada = X_validation
predicciones = model.predict(X_val_escalada)
print(accuracy_score(Y_validation, predicciones))
print(confusion_matrix(Y_validation, predicciones))
print(classification_report(Y_validation, predicciones))


In [ ]:
df_cm = pd.DataFrame(
    confusion_matrix(Y_validation, predicciones),
    columns=np.unique(Y_validation),
    index=np.unique(Y_validation)
)
df_cm.index.name = 'Real'
df_cm.columns.name = 'Predicción'
sns.heatmap(df_cm, cmap='Blues', annot=True, annot_kws={'size': 16})


LDA suele **mejorar** el *recall* frente al árbol en este flujo, pero aún puede quedar un número no nulo de fraudes no detectados en validación. El siguiente bloque aplica **submuestreo aleatorio** de la mayoría para **entrenar** con clases equilibradas; la *razón* es que muchos algoritmos reaccionan mejor cuando la función objetivo no ignora tan cruelmente a la minoría **durante** el ajuste (con el trade-off de perder filas de la clase mayor en entrenamiento).


<a id='5.2'></a>
## 6.2. Balanceo por submuestreo aleatorio (*random under-sampling*)

Se **reduce** el número de transacciones legítimas en el *training* para igualar aproximadamente el número de fraudes, de modo que el clasificador vea *ambas* clases con peso similar al optimizar. **Por qué:** con desbalanceo extremo, el gradiente o los cortes de muchos algoritmos favorecen predecir siempre “no fraude”.

Pasos conceptuales:

1. Cuantificar con `value_counts` cuántas instancias hay por `Class` (conocer el desbalanceo exacto en el *train*).
2. Tomar **todas** las de fraude (en el ejemplo, del orden de cientos) y muestrear **el mismo** número de no fraudes, buscando una proporción 50/50 en el subconjunto de entrenamiento.
3. **Barajar** el subconjunto para no conservar un orden que sesgue la validación cruzada o el orden temporal implícito.

**Coste:** se **pierde** mucha información de transacciones legítimas: el modelo deja de “ver” cientos de miles de normales en el ajuste. El beneficio buscado es **calar** fronteras de decisión en la región de fraude; luego se evalúa otra vez en el conjunto de validación **original** (estratégicamente) para comprobar la detección de fraude en condiciones reales de prevalencia baja.


In [ ]:
Y_train.head()


In [ ]:
df = pd.concat([X_train, Y_train], axis=1)
# Misma cantidad de no fraudes que de fraudes (p. ej. 492 y 492) para equilibrar el entrenamiento
fraud_df = df.loc[df['Class'] == 1]
non_fraud_df = df.loc[df['Class'] == 0][:492]
normal_distributed_df = pd.concat([fraud_df, non_fraud_df])
df_new = normal_distributed_df.sample(frac=1, random_state=42)
Y_train_new = df_new['Class']
X_train_new = df_new.loc[:, dataset.columns != 'Class']
dataset.head()


In [ ]:
print('Distribución de clases en el submuestreo equilibrado')
print(df_new['Class'].value_counts() / len(df_new))
sns.countplot(x='Class', data=df_new)
pyplot.title('Clases equilibradas (submuestreo)', fontsize=14)
pyplot.show()


Con un **marco 50/50** en el entrenamiento, puede volver a usarse la **exactitud** en validación cruzada *dentro* de ese subconjunto, porque allí clases y costes de error a priori son simétricos en el criterio. **Importante:** el **test/validación final** sigue siendo el reservado al principio, con la **prevalencia real** baja, para no engañarnos. La *razón* de volver a `accuracy` en el cruce solo sobre el submuestreo es coherencia con el enfoque equilibrado; el mensaje de negocio se extrae de la matriz de confusión en validación *original*.

Como el número de filas del subconjunto es del orden de **miles o menos** (cientos + cientos), se pueden probar **más** algoritmos, incluida una pequeña red profunda, sin que el tiempo sea prohibitivo (dependiendo del hardware).


In [ ]:
scoring = 'accuracy'


In [ ]:
# Comparación con más algoritmos sobre el entrenamiento equilibrado
models = []
models.append(('RL', LogisticRegression()))
models.append(('LDA', LinearDiscriminantAnalysis()))
models.append(('K-vecinos', KNeighborsClassifier()))
models.append(('CART / árbol', DecisionTreeClassifier()))
models.append(('NB', GaussianNB()))
models.append(('SVM', SVC()))
models.append(('MLP (red)', MLPClassifier()))
models.append(('AdaBoost', AdaBoostClassifier()))
models.append(('GBM', GradientBoostingClassifier()))
models.append(('Bosque aleatorio', RandomForestClassifier()))
models.append(('Extra árboles', ExtraTreesClassifier()))


In [ ]:
# Red profunda (Keras): activar con el flag; construye un MLP sencillo compatible con KerasClassifier
EnableDLModelsFlag = 1
if EnableDLModelsFlag == 1:
    def create_model(neurons=12, activation='relu', learn_rate=0.01, momentum=0):
        model = Sequential()
        model.add(Dense(X_train.shape[1], input_dim=X_train.shape[1], activation=activation))
        model.add(Dense(32, activation=activation))
        model.add(Dense(1, activation='sigmoid'))
        optimizer = SGD(lr=learn_rate, momentum=momentum)
        model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
        return model
    models.append(
        ('DNN', KerasClassifier(
            build_fn=create_model, epochs=50, batch_size=10, verbose=0
        ))
    )


In [ ]:
resultados = []
nombres = []
for nombre, modelo in models:
    kfold = KFold(n_splits=num_folds, random_state=seed)
    puntuaciones_cv = cross_val_score(
        modelo, X_train_new, Y_train_new, cv=kfold, scoring=scoring
    )
    resultados.append(puntuaciones_cv)
    nombres.append(nombre)
    msg = '%s: media=%f (desv.=%f)' % (nombre, puntuaciones_cv.mean(), puntuaciones_cv.std())
    print(msg)


In [ ]:
fig = pyplot.figure()
fig.suptitle('Comparación de algoritmos (submuestreo equilibrado)')
ax = fig.add_subplot(111)
pyplot.boxplot(resultados)
ax.set_xticklabels(nombres)
fig.set_size_inches(8, 4)
pyplot.show()


Si el **gradiente boosting** (`GBM`) obtiene mejores resultados en este barrido, tiene sentido afinar con **`GridSearchCV`** `n_estimators` y `max_depth`: más árboles suelen bajar el sesgo; la profundidad controla el sobreajuste. **Razón:** fijar hiperparámetros sin búsqueda puede dejar de lado una **mejor** generalización; la rejilla se evalúa con la misma métrica y los mismos pliegues.


In [ ]:
# Búsqueda en rejilla: afinar Gradient Boosting
# n_estimators: etapas de boosting (valores mayores a menudo mejor, con coste de tiempo y riesgo de sobreajuste)
# max_depth: profundidad máxima de cada estimador base (árbol); interactúa con el número de variables
n_estimators = [20, 180, 1000]
max_depth = [2, 3, 5]
param_grid = dict(n_estimators=n_estimators, max_depth=max_depth)
modelo_base = GradientBoostingClassifier()
kfold = KFold(n_splits=num_folds, random_state=seed)
grid = GridSearchCV(
    estimator=modelo_base, param_grid=param_grid, scoring=scoring, cv=kfold
)
grid_result = grid.fit(X_train_new, Y_train_new)
print('Mejor: %f con %s' % (grid_result.best_score_, grid_result.best_params_))
medias = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params_ = grid_result.cv_results_['params']
ranks = grid_result.cv_results_['rank_test_score']
for mean, stdev, param, rank in zip(medias, stds, params_, ranks):
    print('#%d %f (%f) con: %r' % (rank, mean, stdev, param))


In [ ]:
# Modelo final en entrenamiento equilibrado (hiperparámetros: ajusta según la rejilla o tu GridSearch)
model = GradientBoostingClassifier(max_depth=5, n_estimators=1000)
model.fit(X_train_new, Y_train_new)


In [ ]:
# Evaluación en el conjunto de validación SIN reequilibrar (misma prevalencia real que al dividir al inicio)
predicciones = model.predict(X_validation)
print(accuracy_score(Y_validation, predicciones))
print(confusion_matrix(Y_validation, predicciones))
print(classification_report(Y_validation, predicciones))


In [ ]:
df_cm = pd.DataFrame(
    confusion_matrix(Y_validation, predicciones),
    columns=np.unique(Y_validation),
    index=np.unique(Y_validation)
)
df_cm.index.name = 'Real'
df_cm.columns.name = 'Predicción'
sns.heatmap(df_cm, cmap='Blues', annot=True, annot_kws={'size': 16})


En el conjunto de validación, el **GBM** afinado puede mostrar un **recall** de fraude muy alto (según la ejecución y el fichero: con el conjunto completo se puede llegar a no dejar fraude sin señal en validación, aunque a costa de más falsas alarmas en legítimas). La *razón* de volver a mirar `classification_report` y la matriz es separar qué mejora es “detectar todo el fraude” y cuánta fricción acepta operaciones o clientes con falsos positivos.


## Conclusión

- **Métrica adecuada:** pasar de la sola *exactitud* a *recall* (y, en un conjunto real, a curvas PR o coste ponderado) **cambia** qué modelos se eligen y detecta el sesgo a la clase mayoritaria.
- **Submuestreo de la mayoría** en *entrenamiento* puede **mejorar** la frontera respecto al fraude, pero el trade-off clásico es perder señal de “no fraude” y **subir falsos positivos** al predecir en el mundo real (validación con prevalencia baja). Hay que sopesar con el coste de negocio: revisar alertas, bloqueo de tarjeta, etc.
- Reproducción: usa la **misma** semilla y el **mismo** volumen de datos que en el análisis de referencia si buscas cifras idénticas; con una **muestra** pequeña, el orden de magnitud de métricas variará, pero el **orden** lógico de pasos (EDA → partición → métrica adecuada → balanceo) se mantiene.
